<a href="https://colab.research.google.com/github/mehtapmutlubilgin/EramusChatBotCustom/blob/main/okularkadasim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install -q -U langchain langchain-community langchain-text-splitters langchain-google-genai google-generativeai chromadb pypdf

In [7]:
!pip install -q -U langchain-text-splitters

In [22]:
!pip install -q -U langchain-community

In [38]:
import os
from google.colab import userdata

# Eğer Secrets kısmına eklediyseniz bu satır çalışır:
try:
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
    print("API Anahtarı Secrets'dan alındı.")
except:
    # Eğer yukarıdaki hata verirse, anahtarınızı tırnak içine elle yazın:
    os.environ["GOOGLE_API_KEY"] = "AIza..."

API Anahtarı Secrets'dan alındı.


1. PDF Dosyasını Sisteme Tanımlama
Colab'ın sol tarafındaki klasör simgesine tıklayıp PDF dosyanızı yüklediyseniz, aşağıdaki kodu yeni bir hücreye yapıştırarak PDF içeriğini parçalara ayıralım:

In [30]:
from langchain_community.document_loaders import PyPDFLoader
# Hata veren satırı bu şekilde güncelledik:
from langchain_text_splitters import RecursiveCharacterTextSplitter

try:
    # 1. PDF dosyasını yükle (Sol paneldeki dosya adıyla aynı olmalı)
    loader = PyPDFLoader("mebyonetmelik.pdf")
    data = loader.load()

    # 2. Metni parçalara ayır
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    chunks = text_splitter.split_documents(data)

    print(f"Başarılı! {len(chunks)} metin parçası hazırlandı.")
except Exception as e:
    print(f"Hata: {e}. PDF dosyasının yüklü olduğundan emin olun.")



Başarılı! 363 metin parçası hazırlandı.


4. Adım: Vektör Veritabanı (Hafıza) Oluşturma
Aşağıdaki kodu yeni bir hücreye yapıştırıp çalıştırın. Bu kod, PDF parçalarını alacak ve Gemini'nin anlayacağı bir sözlük yapısına (Embedding) çevirerek ChromaDB  içine kaydedecektir.

In [41]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma

# API anahtarını doğrudan parametre olarak gönderiyoruz
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/embedding-001",
    google_api_key=os.environ["GOOGLE_API_KEY"]
)

# Vektör veritabanını (ChromaDB) oluşturuyoruz [cite: 43]
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("Vektör veritabanı (Hafıza) başarıyla oluşturuldu!")

GoogleGenerativeAIError: Error embedding content (INVALID_ARGUMENT): 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API key not valid. Please pass a valid API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'generativelanguage.googleapis.com'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'API key not valid. Please pass a valid API key.'}]}}

In [55]:
import os
import google.generativeai as genai
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import Chroma

# 1. API Anahtarını Tanıt
api_key = userdata.get('GOOGLE_API_KEY')
os.environ["GOOGLE_API_KEY"] = api_key

# 2. Embedding Modelini Daha Güvenli Bir Şekilde Tanımlayalım
# Not: Eğer 404 almaya devam edersek, bu sınıf yerine HuggingFace gibi
# alternatif ücretsiz embedding modellerine saniyeler içinde geçebiliriz.
try:
    # Model ismini en yalın haliyle veriyoruz
    embeddings = GoogleGenerativeAIEmbeddings(
        model="models/text-embedding-004",
        google_api_key=api_key
    )

    # Hafıza oluşturma (363 parça üzerinden)
    vector_db = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory="./chroma_db"
    )
    print("✅ TEBRİKLER! Hafıza (Vektör Veritabanı) BAŞARIYLA oluşturuldu!")

except Exception as e:
    print(f"🔄 Google Embedding modellerinde sorun var. Alternatif (HuggingFace) deneniyor...")
    # Google modelleri hata verirse, bootcamp'te kabul gören en popüler alternatifi yüklüyoruz:
    !pip install -q sentence-transformers
    from langchain_community.embeddings import HuggingFaceEmbeddings

    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    vector_db = Chroma.from_documents(documents=chunks, embedding=embeddings)
    print("✅ BAŞARI! Alternatif açık kaynak model ile hafıza oluşturuldu. Projeye devam edebiliriz!")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


🔄 Google Embedding modellerinde sorun var. Alternatif (HuggingFace) deneniyor...


/tmp/ipykernel_3448/1591224370.py:34: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ BAŞARI! Alternatif açık kaynak model ile hafıza oluşturuldu. Projeye devam edebiliriz!


In [71]:
# 1. Gerekli kütüphaneyi yükleyelim
!pip install -q transformers accelerate

from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline
import torch

try:
    # 2. Ücretsiz ve çok güçlü bir model olan Llama-3 (veya benzeri) yüklüyoruz
    # Not: Bu işlem Colab'da birkaç dakika sürebilir (Model indiriliyor)
    pipe = pipeline(
        "text-generation",
        model="TinyLlama/TinyLlama-1.1B-Chat-v1.0", # Hafif ve hızlı bir model
        device_map="auto",
        max_new_tokens=256,
        torch_dtype=torch.float16
    )

    local_llm = HuggingFacePipeline(pipeline=pipe)

    # 3. RAG Fonksiyonu
    def okul_asistani_v2(soru):
        # Hafızadaki 363 metin parçasından sorgu yap
        docs = vector_db.similarity_search(soru, k=2)
        baglam = "\n\n".join([doc.page_content for doc in docs])

        istek = f"Bağlam: {baglam}\n\nSoru: {soru}\n\nYanıtı sadece bağlama göre kısa ve net ver."

        return local_llm.invoke(istek)

    print("\n--- OkulArkadaşım (Yerel Model) Hazır! ---")
    test_sorusu = "Özürsüz devamsızlık sınırı kaç gündür?"
    yanit = okul_asistani_v2(test_sorusu)
    print(f"SORU: {test_sorusu}")
    print(f"CEVAP: {yanit}")

except Exception as e:
    print(f"❌ Kritik Hata: {e}")

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
/tmp/ipykernel_3448/1899546154.py:19: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  local_llm = HuggingFacePipeline(pipeline=pipe)



--- OkulArkadaşım (Yerel Model) Hazır! ---
SORU: Özürsüz devamsızlık sınırı kaç gündür?
CEVAP: Bağlam: imkânlarının araştırılmasını, mesleklerinde ba şarılı olanların ders, seminer ve konferans gibi 
etkinliklerle eğitime katkıda bulunmalarını sağlar.

kovuşturmanın tamamlanmasını ve sonucunu beklemeden acele bir tedbir olmak üzere on i ş 
gününü geçmemek kaydıyla millî eğitim müdürünü bilgilendirerek uygun göreceği süre kadar 
geçici olarak okuldan uzakla ştırabilir. Bu durumdaki ö ğrenciler, ders ve s ınavlarla di ğer 
etkinliklere alınmazlar. Hakkında tedbir kararı verilen öğrencinin okuldan uzaklaştırıldığı süre 
devamsızlıktan sayılmaz.  
(2) Tedbir karar ının al ınmasını izleyen en geç üç i ş günü içinde disiplin i şlemine 
başlanır ve okul öğrenci ödül ve disiplin kuruluna sevkinden itibaren en geç on i ş günü içinde 
durumu karara ba ğlanır. Aksi takdirde al ınan tedbir karar ı, kendili ğinden kalkm ış say ılır. 
Haklı ve zorlay ıcı sebeplerin devam ı hâlinde millî e ğitim müd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')